# References từ Semantic Scholar

Lấy references theo từng paper và phân trang. Working data giữ toàn bộ references; course export chỉ giữ cạnh có arXiv ID. `crawl_status.parquet` lưu trạng thái truy vấn, kể cả kết quả rỗng.

BibTeX fallback và metadata bổ sung được ghi nguồn riêng. Ngày công bố S2 chưa được coi là ngày nộp arXiv đã xác minh.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from crawler.reporting import load_m1, run_table
m1 = load_m1()
papers, refs, versions, observations = (m1[k] for k in ('papers','references','versions','observations'))
print('Input:', m1['directory'])
print('Phạm vi:', len(papers), 'paper;', len(versions), 'version;', len(refs), 'working references')
print('Provenance:', m1['provenance'])


In [ ]:
display(refs['source'].fillna('unknown').value_counts().to_frame('reference_rows'))
display(refs.isna().mean().to_frame('missing_fraction'))
if not observations.empty:
    display(observations['reference_status'].fillna('unknown').value_counts().to_frame('papers'))
    known = observations[observations['reference_status'].ne('unknown')]
    print('API complete/confirmed empty rate:', known['reference_status'].isin(['success','empty']).mean() if len(known) else None)
    print('Request delta tổng từ sidecar hiện có:', observations['requests'].sum(min_count=1))
    valid_ids = observations[observations['reference_status'].isin(['success','empty','bib_fallback'])]['arxiv_id']
    per_paper = refs.groupby('arxiv_id').size().reindex(valid_ids, fill_value=0)
    print('Reference rows trung bình trên tập status hợp lệ:', per_paper.mean() if len(per_paper) else None)
else:
    print('Thiếu crawl_status.parquet: chưa tính được API success rate hoặc phân loại paper rỗng.')
if len(refs):
    refs['source'].fillna('unknown').value_counts().plot.bar(title='Nguồn của các reference rows', ylabel='Số dòng')
    plt.tight_layout(); plt.show()


## Giới hạn

Tính trung bình trên tập paper có trạng thái đã xác minh; lỗi API được báo riêng. `requests` trong sidecar là số request của lần truy vấn đã lưu; tổng qua nhiều run lấy từ run logs.
